# ETM5800 Assignment 1 Notebook

## Best Buy MacBook Air 13-inch M3 Public Review Dataset

This notebook documents the collection, preparation, and exploratory review of a public customer-review dataset for the Apple MacBook Air 13-inch M3 listing on Best Buy.

It is designed to support Assignment 1 by showing:

1. the data source and business context,
2. the main preprocessing workflow,
3. core dataset diagnostics,
4. basic exploratory outputs that align with the written report.


## Colab Setup Note

Upload the project folder to Google Drive first, then run the next two cells in Colab.

In [ ]:
# Run this cell in Colab first.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

project_dir = Path('/content/drive/MyDrive/data')
if not (project_dir / 'data' / 'cleaned_reviews.csv').exists():
    project_dir = Path.cwd().parent if Path.cwd().name == 'output' else Path.cwd()

data_dir = project_dir / 'data'
diagnostics = json.loads((data_dir / 'diagnostics.json').read_text(encoding='utf-8'))
raw_df = pd.read_csv(data_dir / 'raw_reviews.csv')
clean_df = pd.read_csv(data_dir / 'cleaned_reviews.csv')

clean_df['review_date_parsed'] = pd.to_datetime(clean_df['review_date'], format='%b %d, %Y %I:%M %p', errors='coerce')
clean_df['review_month'] = clean_df['review_date_parsed'].dt.to_period('M').astype(str)

print('Raw reviews:', len(raw_df))
print('Cleaned reviews:', len(clean_df))
print('2,000 target met:', diagnostics.get('two_thousand_target_met'))

## Business Context

The dataset is relevant to a business decision problem because public reviews reveal which product attributes matter most to customers. For this listing, likely themes include battery life, portability, speed, and use for study or work. These themes can support later analysis for product positioning, customer communication, and service monitoring.

In [ ]:
summary = {
    'platform': diagnostics.get('platform'),
    'market': diagnostics.get('market'),
    'canonical_product_topic': diagnostics.get('canonical_product_topic'),
    'raw_reviews': diagnostics.get('total_raw_reviews_collected'),
    'cleaned_reviews': diagnostics.get('total_cleaned_reviews_retained'),
    'duplicate_removed': diagnostics.get('duplicate_count_removed'),
    'short_or_empty_removed': diagnostics.get('short_or_empty_review_count_removed'),
    'selected_product_ids': diagnostics.get('selected_product_ids'),
}
summary

## Schema Check

The assignment requires a clearly labelled final text column and supporting metadata. The cells below confirm the available fields.

In [ ]:
print('Raw columns:')
for column in raw_df.columns:
    print('-', column)

print('\nCleaned columns:')
for column in clean_df.columns:
    print('-', column)

## Example Rows

This provides a quick sense of the stored text, rating, and language fields.

In [ ]:
clean_df[['review_date', 'rating_stars', 'language_guess', 'review_text_raw', 'text']].head(5)

## Data Quality Checks

The cleaned dataset should preserve useful text while avoiding missing final text values.

In [ ]:
quality_checks = {
    'rows_with_missing_final_text': int(clean_df['text'].fillna('').str.strip().eq('').sum()),
    'rows_with_missing_masked_name': int(clean_df['reviewer_name_masked'].fillna('').str.strip().eq('').sum()),
    'min_word_count': int(clean_df['text_word_count'].min()),
    'median_word_count': float(clean_df['text_word_count'].median()),
    'mean_word_count': round(float(clean_df['text_word_count'].mean()), 2),
    'date_min': str(clean_df['review_date_parsed'].min()),
    'date_max': str(clean_df['review_date_parsed'].max()),
}
quality_checks

## Core Descriptive Statistics

In [ ]:
rating_counts = clean_df['rating_stars'].astype(str).value_counts().sort_index()
language_counts = clean_df['language_guess'].fillna('unknown').value_counts()

print('Rating distribution:')
print(rating_counts.to_dict())

print('\nLanguage distribution:')
print(language_counts.to_dict())

## Reproduced Visual Checks

The next cells reproduce a few lightweight visuals directly in the notebook so the Colab version remains self-contained.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=rating_counts.index, y=rating_counts.values, hue=rating_counts.index, legend=False, palette='crest', ax=ax)
ax.set_title('Rating distribution')
ax.set_xlabel('Rating stars')
ax.set_ylabel('Review count')
plt.show()

In [ ]:
monthly_counts = clean_df.groupby('review_month').size().reset_index(name='review_count')
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=monthly_counts, x='review_month', y='review_count', marker='o', ax=ax)
ax.set_title('Monthly review volume')
ax.set_xlabel('Review month')
ax.set_ylabel('Review count')
ax.tick_params(axis='x', rotation=60)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(clean_df['text_word_count'], bins=35, kde=True, color='#2a9d8f', ax=ax)
ax.set_title('Review length distribution')
ax.set_xlabel('Words per review')
ax.set_ylabel('Count')
plt.show()

## Short Notes

A few simple patterns are already visible:

- Most reviews are positive.
- Review text often mentions battery life, speed, and study or work use.
- The dataset is ready for later text analysis.